In [2]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.5 MB/s eta 0:00:00


In [4]:
!pip install gradio

In [5]:
import os
import re
import unicodedata
from pathlib import Path
import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import spacy
import textstat
from collections import Counter
import gradio as gr

# Install required packages (uncomment if needed)
# !pip install gradio sentence-transformers spacy textstat joblib
# !python -m spacy download en_core_web_sm

# ============================================================================
# CONFIGURATION
# ============================================================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# For Google Colab with Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ARTIFACT_DIR = Path('/content/drive/MyDrive/NLP/artifacts')
except:
    # For local or Hugging Face Spaces
    ARTIFACT_DIR = Path('./artifacts')

EMB_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

# ============================================================================
# TEXT PREPROCESSING
# ============================================================================

def normalize_unicode_and_space(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace('\u2013', '-').replace('\u2014', ' - ')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

punct_space_re = re.compile(r"\s*([,.;:!?()\[\]{}\-—\"'""''])\s*")

def fix_punctuation_spacing(text: str) -> str:
    if not text:
        return ""
    text = punct_space_re.sub(r" \1 ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def basic_clean(text: str) -> str:
    t = normalize_unicode_and_space(text)
    t = fix_punctuation_spacing(t)
    return t

def sentence_split(text: str, nlp):
    text = normalize_unicode_and_space(text)
    try:
        doc = nlp(text)
        sents = [s.text.strip() for s in doc.sents]
        if sents:
            return sents
    except Exception:
        pass
    sents = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sents if s.strip()]

def tokenize_with_spacy(text: str, nlp):
    text = basic_clean(text)
    doc = nlp(text)
    tokens = [tok for tok in doc if not tok.is_space]
    words = [tok.text for tok in tokens if tok.is_alpha]
    return tokens, words

def linguistic_annotations(text: str, nlp):
    text = basic_clean(text)
    doc = nlp(text)
    pos_dep = [(tok.text, tok.lemma_, tok.pos_, tok.tag_, tok.dep_) for tok in doc]
    ents = [(ent.text, ent.label_) for ent in doc.ents] if hasattr(doc, 'ents') else []
    sents = [s.text for s in doc.sents]
    return {'pos_dep': pos_dep, 'entities': ents, 'sents': sents}

def count_dialogue_markers(text: str) -> int:
    if not text:
        return 0
    return text.count('"') + text.count("'") + text.count('—') + text.count('-')

# ============================================================================
# FEATURE EXTRACTOR
# ============================================================================

class StylometricFeatureExtractor:
    def __init__(self, nlp, enable_readability=True):
        self.nlp = nlp
        self.enable_readability = enable_readability

    def _approx_syllable_count(self, w):
        return max(1, len(re.findall(r'[aeiouyAEIOUY]+', w)))

    def extract_all_features(self, text):
        text_orig = "" if text is None else str(text)
        text = basic_clean(text_orig)

        features = {}
        sents = sentence_split(text, self.nlp)
        tokens, words = tokenize_with_spacy(text, self.nlp)
        num_words = len(words)

        features['char_count'] = len(text)
        features['word_count'] = num_words
        features['sentence_count'] = max(1, len(sents))
        features['avg_word_length'] = float(np.mean([len(w) for w in words])) if words else 0.0
        features['avg_sentence_length'] = float(num_words) / features['sentence_count'] if features['sentence_count'] > 0 else 0.0

        sent_word_counts = []
        try:
            doc = self.nlp(text)
            for s in doc.sents:
                sent_word_counts.append(len([t for t in s if t.is_alpha]))
        except Exception:
            sent_word_counts = [len(s.split()) for s in sents]

        if sent_word_counts:
            features['sent_len_mean'] = float(np.mean(sent_word_counts))
            features['sent_len_std'] = float(np.std(sent_word_counts))
            features['sent_len_max'] = int(np.max(sent_word_counts))
        else:
            features['sent_len_mean'] = features['sent_len_std'] = 0.0
            features['sent_len_max'] = 0

        words_lower = [w.lower() for w in words]
        if words_lower:
            freq = Counter(words_lower)
            features['type_token_ratio'] = len(set(words_lower)) / len(words_lower)
            features['hapax_legomena_ratio'] = sum(1 for v in freq.values() if v == 1) / len(words_lower)
            try:
                M1 = len(words_lower)
                features['yules_k'] = (1_000_000 * (sum(v*v for v in freq.values()) - M1)) / (M1*M1)
            except Exception:
                features['yules_k'] = 0.0
        else:
            features['type_token_ratio'] = 0.0
            features['hapax_legomena_ratio'] = 0.0
            features['yules_k'] = 0.0

        denom = max(1, features['char_count'])
        features['comma_freq'] = text.count(',') / denom * 1000
        features['period_freq'] = text.count('.') / denom * 1000
        features['exclamation_freq'] = text.count('!') / denom * 1000
        features['question_freq'] = text.count('?') / denom * 1000
        features['quote_freq'] = (text.count('"') + text.count("'")) / denom * 1000

        features['dialogue_markers'] = count_dialogue_markers(text) / max(1, num_words)
        features['first_person'] = sum(1 for w in words_lower if w in ['i','me','my','we','us','our']) / max(1, len(words_lower))

        try:
            ann = linguistic_annotations(text, self.nlp)
            pos_dep = ann['pos_dep']
            total_pos = max(1, len(pos_dep))
            pos_counts = Counter([p[2] for p in pos_dep])
            for p in ['NOUN','VERB','ADJ','ADV','PRON','ADP','CONJ','DET','NUM','PUNCT']:
                features[f'pos_{p}'] = pos_counts.get(p, 0) / total_pos
            deps = [p[4] for p in pos_dep]
            features['passive_ratio'] = (deps.count('auxpass') + sum(1 for i,p in enumerate(pos_dep[:-1]) if p[2]=='AUX' and pos_dep[i+1][3]=='VBN')) / total_pos

            try:
                doc_full = self.nlp(text)
                depths = []
                for tok in doc_full:
                    depth = 0
                    node = tok
                    while node.head is not node:
                        depth += 1
                        node = node.head
                        if depth > 200:
                            break
                    depths.append(depth)
                features['parse_depth_mean'] = float(np.mean(depths)) if depths else 0.0
                features['parse_depth_max'] = int(np.max(depths)) if depths else 0
            except Exception:
                features['parse_depth_mean'] = 0.0
                features['parse_depth_max'] = 0
        except Exception:
            for p in ['NOUN','VERB','ADJ','ADV','PRON','ADP','CONJ','DET','NUM','PUNCT']:
                features[f'pos_{p}'] = 0.0
            features['passive_ratio'] = 0.0
            features['parse_depth_mean'] = 0.0
            features['parse_depth_max'] = 0

        if self.enable_readability:
            try:
                features['flesch_reading_ease'] = textstat.flesch_reading_ease(text)
                features['flesch_kincaid_grade'] = textstat.flesch_kincaid_grade(text)
            except Exception:
                syllables = sum(self._approx_syllable_count(w) for w in words_lower) if words_lower else 0
                features['flesch_reading_ease'] = 206.835 - 1.015 * (num_words / max(1, features['sentence_count'])) - 84.6 * (syllables / max(1, num_words))
                features['flesch_kincaid_grade'] = 0.0

        return features

# ============================================================================
# MODEL
# ============================================================================

class FusionClassifier(nn.Module):
    def __init__(self, stylo_dim, emb_dim, hidden=512, num_classes=2, dropout=0.3):
        super().__init__()
        self.bn = nn.BatchNorm1d(stylo_dim + emb_dim)
        self.fc1 = nn.Linear(stylo_dim + emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden//2)
        self.out = nn.Linear(hidden//2, num_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, stylo, emb):
        x = torch.cat([stylo, emb], dim=1)
        x = self.bn(x)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        x = self.drop(x)
        return self.out(x)

# ============================================================================
# LOAD MODEL AND ARTIFACTS
# ============================================================================

print("Loading model and artifacts...")

# Load spaCy
nlp = spacy.load('en_core_web_sm')

# Load artifacts
stylo_cols = joblib.load(ARTIFACT_DIR / 'stylo_columns.pkl')
scaler = joblib.load(ARTIFACT_DIR / 'stylo_scaler.joblib')
vt = joblib.load(ARTIFACT_DIR / 'stylo_var_threshold.joblib')
label_encoder = joblib.load(ARTIFACT_DIR / 'label_encoder.joblib')
num_classes = len(label_encoder.classes_)

# Load sentence transformer
embedder = SentenceTransformer(EMB_MODEL)
embedder = embedder.to(DEVICE)

# Determine dimensions
sample_stylo = np.zeros((1, len(stylo_cols)))
sample_stylo = scaler.transform(sample_stylo)
sample_stylo = vt.transform(sample_stylo)
stylo_dim = sample_stylo.shape[1]

sample_emb = embedder.encode(["test"])
emb_dim = sample_emb.shape[1]

# Load model
model = FusionClassifier(
    stylo_dim=stylo_dim,
    emb_dim=emb_dim,
    hidden=512,
    num_classes=num_classes,
    dropout=0.3
).to(DEVICE)

ckpt = torch.load(ARTIFACT_DIR / 'best_fusion.pt',
                  map_location=DEVICE,
                  weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Create feature extractor
extractor = StylometricFeatureExtractor(nlp, enable_readability=True)

print(f"✓ Model loaded! Classes: {list(label_encoder.classes_)}")

# ============================================================================
# PREDICTION FUNCTION FOR GRADIO
# ============================================================================

def predict_style(text):

    if not text or len(text.strip()) < 5:
        return "⚠️ Please enter at least 5 characters of text.", {}

    try:
        # Extract features
        feats = extractor.extract_all_features(text)
        stylo_vec = np.array([feats.get(c, 0.0) for c in stylo_cols]).reshape(1, -1)
        stylo_vec = scaler.transform(stylo_vec)
        stylo_vec = vt.transform(stylo_vec)

        emb = embedder.encode([text])

        with torch.no_grad():
            sty_t = torch.tensor(stylo_vec).float().to(DEVICE)
            emb_t = torch.tensor(emb).float().to(DEVICE)
            logits = model(sty_t, emb_t)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]
            pred = logits.argmax(1).cpu().numpy()[0]

        predicted_label = label_encoder.inverse_transform([pred])[0]
        confidence = float(probs[pred])

        prob_dict = {label: float(prob)
                    for label, prob in zip(label_encoder.classes_, probs)}

        output = f"""
## 🎯 Prediction Results

**Predicted Style:** `{predicted_label}`
**Confidence:** {confidence:.2%}

### 📊 Probability Distribution:
"""
        for label, prob in sorted(prob_dict.items(), key=lambda x: x[1], reverse=True):
            bar_length = int(prob * 30)
            bar = "█" * bar_length + "░" * (30 - bar_length)
            output += f"\n- **{label}**: {bar} `{prob:.2%}`"

        return output, prob_dict

    except Exception as e:
        return f"❌ Error during prediction: {str(e)}", {}


examples = [
    ["In the evening, she whispered to the moon about the sea of memories."],
    ["The quantum computer performed calculations at unprecedented speeds, revolutionizing computational science."],
    ["Once upon a time, in a land far away, there lived a brave knight who sought to rescue a princess."],
    ["The stock market rallied today as investors reacted positively to the Federal Reserve's decision."],
    ["To be or not to be, that is the question. Whether 'tis nobler in the mind to suffer the slings and arrows."]
]

with gr.Blocks(theme=gr.themes.Soft(), title="Text Style Classifier") as demo:
    gr.Markdown(
        """
        # 📝 Text Style Classifier

        This AI model analyzes the stylistic features of your text and predicts its writing style.
        The model uses both **stylometric features** (sentence length, vocabulary richness, punctuation patterns)
        and **semantic embeddings** to make accurate predictions.

        ### How to use:
        1. Enter or paste your text in the box below
        2. Click "Classify Style" to see the prediction
        3. View the confidence scores for each style category
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(
                label="Enter your text here",
                placeholder="Type or paste your text here... (minimum 5 characters)",
                lines=8,
                max_lines=20
            )

            with gr.Row():
                clear_btn = gr.Button("Clear", variant="secondary")
                submit_btn = gr.Button("🔍 Classify Style", variant="primary")

        with gr.Column(scale=2):
            output_text = gr.Markdown(label="Results")
            output_probs = gr.Label(label="Confidence Scores", num_top_classes=num_classes)

    gr.Markdown("### 📚 Example Texts")
    gr.Examples(
        examples=examples,
        inputs=input_text,
        label="Click an example to try it out"
    )

    gr.Markdown(
        """
        ---
        **Note:** The model analyzes linguistic patterns including:
        - Sentence structure and length
        - Vocabulary diversity
        - Punctuation usage
        - Part-of-speech patterns
        - Readability metrics
        """
    )

    submit_btn.click(
        fn=predict_style,
        inputs=input_text,
        outputs=[output_text, output_probs]
    )

    clear_btn.click(
        fn=lambda: ("", "", {}),
        outputs=[input_text, output_text, output_probs]
    )

    input_text.submit(
        fn=predict_style,
        inputs=input_text,
        outputs=[output_text, output_probs]
    )


if __name__ == "__main__":

    demo.launch(
        share=True,
        server_name="0.0.0.0",
        server_port=7860,
        show_error=True
    )

<>:61: SyntaxWarning: invalid escape sequence '\s'
<>:61: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-455808288.py:61: SyntaxWarning: invalid escape sequence '\s'
  punct_space_re = re.compile(r"\s*([,.;:!?()\[\]{}\-—\"'""''])\s*")


Using device: cpu
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model and artifacts...
✓ Model loaded! Classes: ['Academic', 'Normal', 'journalistic', 'narrative', 'philosophical', 'poetic']
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b50448c5e6e6d6be06.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
